# 🍎 Food Freshness Detection Using Deep Learning
**Fresh vs Rotten fruit classification with MobileNetV2 (transfer learning) + Gradio app**

**How to use this notebook (Google Colab)**
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all` (or run each cell top to bottom). **No dataset upload is needed**. Cell 3 downloads it automatically.
3. The last cell launches a Gradio web app (click the public link it prints).

**Dataset:** *Fresh and Rotten Fruits Dataset for Machine-Based Evaluation of Fruit Quality* (Sultana, Jahan & Uddin, 2022, *Data in Brief*), hosted on Hugging Face as `Project-AgML/fresh_rotten_fruit_classification` (licence: CC BY 4.0). We use its **`raw` subset: 3,200 original photos (1,600 fresh + 1,600 rotten) of 8 fruit types**. The *augmented* subset is deliberately NOT used, because augmented copies of one photo could land in both train and test (data leakage).

**Pipeline:** download → export to `fresh/` and `rotten/` folders → stratified train/val/test split → augmentation (train only) → MobileNetV2 (frozen) → fine-tune → evaluate → save → Gradio.

## 1. Install packages

In [ ]:
!pip install -q gradio datasets

## 2. Imports and settings

In [ ]:
import os, json, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)       # MobileNetV2 input size
BATCH_SIZE = 32
EPOCHS_FROZEN = 10          # phase 1: train only the new classifier head
EPOCHS_FINETUNE = 8         # phase 2: fine-tune the top MobileNetV2 layers
CLASS_NAMES = ["Fresh", "Rotten"]   # label 0 = Fresh, label 1 = Rotten

# Optional: restrict to the 3 fruits from the assignment brief.
# Set to None to use all 8 fruit types in the dataset.
FRUITS = None               # e.g. ["apple", "banana", "orange"]

RESULTS_DIR = "/content/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU found:", tf.config.list_physical_devices("GPU"))

## 3. Download the dataset automatically (no login, no manual upload)
The dataset is downloaded from the Hugging Face Hub. Images are then saved into this folder structure (resized to 224×224):

```
/content/fruit_data/
├── fresh/    apple_00000.jpg, banana_00012.jpg, ...
└── rotten/   apple_01600.jpg, orange_01753.jpg, ...
```
The fruit type is kept in the file name and used only to make the train/val/test split balanced.

In [ ]:
from datasets import load_dataset

# "raw" = the 3,200 original images (NOT the augmented copies)
hf_ds = load_dataset("Project-AgML/fresh_rotten_fruit_classification", "raw", split="train")
print(hf_ds)

label_names = hf_ds.features["label"].names
print("Label names in dataset:", label_names)

In [ ]:
DATA_DIR = "/content/fruit_data"
rows, skipped = [], 0

for i, ex in enumerate(tqdm(hf_ds, desc="Exporting images")):
    state = label_names[ex["label"]].lower()          # 'fresh' or 'rotten'
    fruit = str(ex["fruit_type"]).lower()             # e.g. 'apple'
    folder = os.path.join(DATA_DIR, state)
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, f"{fruit}_{i:05d}.jpg")
    try:
        if not os.path.exists(path):
            ex["image"].convert("RGB").resize(IMG_SIZE).save(path, "JPEG", quality=95)
        rows.append({"path": path, "state": state, "fruit": fruit})
    except Exception as e:                            # skip unreadable images
        skipped += 1

df = pd.DataFrame(rows)
df["label"] = (df["state"] == "rotten").astype(int)   # Fresh=0, Rotten=1

if FRUITS is not None:
    df = df[df["fruit"].isin(FRUITS)].reset_index(drop=True)

df["strata"] = df["fruit"] + "_" + df["state"]
print(f"Images exported: {len(df)} | skipped: {skipped}")
print(df.groupby(["fruit", "state"]).size().unstack(fill_value=0))

In [ ]:
# Look at a few samples
sample = df.sample(8, random_state=SEED)
plt.figure(figsize=(12, 6))
for k, (_, r) in enumerate(sample.iterrows()):
    plt.subplot(2, 4, k + 1)
    plt.imshow(Image.open(r.path)); plt.axis("off")
    plt.title(f"{r.state} {r.fruit}")
plt.suptitle("Random samples from the dataset"); plt.show()

## 4. Train / validation / test split (no data leakage)
We split the **file list first** (70% / 15% / 15%), stratified by fruit + freshness, *before* any augmentation. A check confirms no image appears in two splits.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["strata"], random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df["strata"], random_state=SEED)

assert not (set(train_df.path) & set(val_df.path))
assert not (set(train_df.path) & set(test_df.path))
assert not (set(val_df.path) & set(test_df.path))

for name, d in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name:10s}: {len(d):5d} images | Fresh={(d.label==0).sum()}  Rotten={(d.label==1).sum()}")

## 5. Preprocessing and augmentation (augmentation on training data only)
* Images are resized to 224×224.
* Pixel scaling to **[-1, 1]** (what MobileNetV2 expects) is built **inside the model** (next section), so training and prediction always use identical preprocessing.
* Augmentation (flip, rotation, zoom, contrast) is applied to the **training set only**.

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name="augmentation")

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)               # float32, values 0-255
    img.set_shape((IMG_SIZE[0], IMG_SIZE[1], 3))
    return img, label

def make_dataset(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (frame["path"].values, frame["label"].values.astype("float32")))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE)
    if training:                                        # augmentation ONLY for training
        ds = ds.map(lambda x, y: (augment(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)          # not shuffled -> order matches test_df

imgs, labs = next(iter(train_ds))
plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(imgs[i].numpy().astype("uint8")); plt.axis("off")
    plt.title(CLASS_NAMES[int(labs[i])])
plt.suptitle("Augmented training samples"); plt.show()

## 6. Build the model (MobileNetV2 transfer learning)
`Input → Rescaling([-1,1]) → MobileNetV2 (ImageNet weights, frozen) → GlobalAveragePooling → Dropout(0.3) → Dense(1, sigmoid)`

The single output is **P(Rotten)**. A value ≥ 0.5 means *Rotten*, otherwise *Fresh*.

In [ ]:
def build_model():
    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = layers.Rescaling(1/127.5, offset=-1, name="mobilenet_preprocess")(inputs)   # 0-255 -> [-1, 1]
    base = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
    base.trainable = False                       # freeze pretrained layers
    x = base(x, training=False)                  # keep BatchNorm in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs), base

model, base_model = build_model()
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## 7. Phase 1 – train the new classifier head (base frozen)

In [ ]:
n0, n1 = (train_df.label == 0).sum(), (train_df.label == 1).sum()
class_weight = {0: len(train_df) / (2 * n0), 1: len(train_df) / (2 * n1)}
print("Class weights:", class_weight)

callbacks_p1 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("best_phase1.keras", monitor="val_loss", save_best_only=True),
]
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FROZEN,
                  callbacks=callbacks_p1, class_weight=class_weight)

## 8. Phase 2 – fine-tune the top 30 MobileNetV2 layers
A very small learning rate (1e-5) is used so the pretrained features are adjusted gently. BatchNorm layers stay frozen for stability.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="binary_crossentropy", metrics=["accuracy"])

callbacks_p2 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("best_phase2.keras", monitor="val_loss", save_best_only=True),
]
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE,
                  callbacks=callbacks_p2, class_weight=class_weight)

## 9. Pick the better model using **validation** data (never the test set)

In [ ]:
m1 = keras.models.load_model("best_phase1.keras")
m2 = keras.models.load_model("best_phase2.keras")
v1 = m1.evaluate(val_ds, verbose=0)
v2 = m2.evaluate(val_ds, verbose=0)
print(f"Phase 1 (frozen)     -> val_loss={v1[0]:.4f}  val_acc={v1[1]:.4f}")
print(f"Phase 2 (fine-tuned) -> val_loss={v2[0]:.4f}  val_acc={v2[1]:.4f}")

final_model = m2 if v2[0] <= v1[0] else m1
print("Selected model:", "Phase 2 (fine-tuned)" if final_model is m2 else "Phase 1 (frozen base)")

## 10. Training and validation curves

In [ ]:
acc      = hist1.history["accuracy"]      + hist2.history["accuracy"]
val_acc  = hist1.history["val_accuracy"]  + hist2.history["val_accuracy"]
loss     = hist1.history["loss"]          + hist2.history["loss"]
val_loss = hist1.history["val_loss"]      + hist2.history["val_loss"]
split_at = len(hist1.history["accuracy"])

plt.figure(figsize=(12, 4))
for i, (tr, va, title) in enumerate([(acc, val_acc, "Accuracy"), (loss, val_loss, "Loss")]):
    plt.subplot(1, 2, i + 1)
    plt.plot(tr, label="Train"); plt.plot(va, label="Validation")
    plt.axvline(split_at - 1, color="gray", linestyle="--", label="Fine-tuning starts")
    plt.title(title); plt.xlabel("Epoch"); plt.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/training_curves.png", dpi=150)
plt.show()

## 11. Final evaluation on the untouched TEST set

In [ ]:
y_prob = final_model.predict(test_ds, verbose=0).ravel()
y_true = test_df["label"].values
y_pred = (y_prob >= 0.5).astype(int)

metrics = {
    "test_images": int(len(y_true)),
    "accuracy":  float(accuracy_score(y_true, y_pred)),
    "precision_rotten": float(precision_score(y_true, y_pred)),
    "recall_rotten":    float(recall_score(y_true, y_pred)),
    "f1_rotten":        float(f1_score(y_true, y_pred)),
    "roc_auc":          float(roc_auc_score(y_true, y_prob)),
}
for k, v in metrics.items():
    print(f"{k:18s}: {v:.4f}" if isinstance(v, float) else f"{k:18s}: {v}")

print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_true, y_pred)
metrics["confusion_matrix"] = cm.tolist()
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix (Test Set)")
plt.savefig(f"{RESULTS_DIR}/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# Accuracy for each fruit type (useful discussion point for the report)
per_fruit = (test_df.assign(correct=(y_pred == y_true))
                    .groupby("fruit")["correct"].agg(["mean", "count"])
                    .rename(columns={"mean": "accuracy", "count": "n_test_images"}).round(4))
print("\nAccuracy per fruit type (test set):")
print(per_fruit)

# ROC curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.figure(figsize=(5, 4.5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {metrics['roc_auc']:.4f}")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Test Set)"); plt.legend()
plt.savefig(f"{RESULTS_DIR}/roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()

with open(f"{RESULTS_DIR}/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
per_fruit.to_csv(f"{RESULTS_DIR}/per_fruit_accuracy.csv")

## 12. Save the trained model
The saved file contains the preprocessing layer too, so it accepts raw 0-255 images.

In [ ]:
final_model.save("food_freshness_model.keras")
with open("class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f)
print("Saved: food_freshness_model.keras and class_names.json")
print("TensorFlow version used for training:", tf.__version__, "<- use the same version in requirements.txt for Streamlit deployment")

# Optional – keep a copy in Google Drive so you never need to retrain:
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copy("food_freshness_model.keras", "/content/drive/MyDrive/food_freshness_model.keras")

## 13. Reuse the saved model WITHOUT retraining
In a new Colab session: run Cells 1–2, upload `food_freshness_model.keras` (left sidebar → Files, or from Drive), then run this cell and the Gradio cell. No dataset or training is needed.

In [ ]:
model = keras.models.load_model("food_freshness_model.keras")

def predict_path(image_path):
    img = Image.open(image_path).convert("RGB").resize(IMG_SIZE)
    arr = np.expand_dims(np.array(img, dtype="float32"), 0)    # raw 0-255; scaling is inside the model
    p_rotten = float(model.predict(arr, verbose=0)[0][0])
    label = "Rotten" if p_rotten >= 0.5 else "Fresh"
    return label, max(p_rotten, 1 - p_rotten)

# Quick sanity check on one test image
row = test_df.iloc[0]
lab, conf = predict_path(row.path)
print(f"File: {row.path}\nTrue: {CLASS_NAMES[row.label]} | Predicted: {lab} ({conf*100:.1f}%)")

## 14. Gradio web app
Upload a fruit photo → get *Fresh* or *Rotten* with a confidence score. Invalid or unreadable files show a clear error message instead of crashing.

In [ ]:
import gradio as gr

def predict_freshness(image_path):
    if image_path is None:
        return None, "⚠️ Please upload an image first."
    try:
        img = Image.open(image_path).convert("RGB").resize(IMG_SIZE)
        arr = np.expand_dims(np.array(img, dtype="float32"), 0)
        p_rotten = float(model.predict(arr, verbose=0)[0][0])
    except Exception:
        return None, "❌ This file could not be processed. Please upload a valid JPG or PNG image."

    label = "Rotten" if p_rotten >= 0.5 else "Fresh"
    conf = max(p_rotten, 1 - p_rotten)
    scores = {"Fresh": 1 - p_rotten, "Rotten": p_rotten}
    msg = f"### Prediction: {label}\n**Confidence: {conf*100:.1f}%**"
    if conf < 0.70:
        msg += "\n\n⚠️ Low confidence – the photo may be unclear, or the item may not be one of the fruits the model was trained on."
    return scores, msg

with gr.Blocks(title="Food Freshness Detection") as demo:
    gr.Markdown("# 🍎 Food Freshness Detection Using Deep Learning")
    gr.Markdown("Upload a clear photo of a fruit (apple, banana, orange, grape, guava, jujube, pomegranate or strawberry). "
                "A **MobileNetV2** model (transfer learning) predicts whether it is **Fresh** or **Rotten**. "
                "The model only knows these fruits, so other objects may give unreliable results.")
    with gr.Row():
        inp = gr.Image(type="filepath", label="Upload fruit image")
        with gr.Column():
            out_label = gr.Label(num_top_classes=2, label="Class probabilities")
            out_msg = gr.Markdown()
    gr.Button("Predict", variant="primary").click(predict_freshness, inp, [out_label, out_msg])

demo.launch(share=True)

## 15. (Optional) Download your results for the report
Zips the model, graphs, `metrics.json` and the per-fruit table so you can paste the **real** numbers and figures into your report.

**For the Streamlit/GitHub deployment:** copy `food_freshness_model.keras` and `metrics.json` from this zip into the `model/` folder of the repo.

In [ ]:
shutil.copy("food_freshness_model.keras", f"{RESULTS_DIR}/food_freshness_model.keras")
shutil.make_archive("/content/project_outputs", "zip", RESULTS_DIR)
try:
    from google.colab import files
    files.download("/content/project_outputs.zip")
except Exception as e:
    print("Download manually from the Files panel: /content/project_outputs.zip")